# Reproducing *Near-Linear-Time Bilevel Congestion Network Design*

Companion notebook for the DNLF paper. Regenerates the headline numbers with timing on the Sioux Falls
benchmark, then shows the size-independence of the algorithm. For the full irregular-graph crossover vs.
a direct factorization, run `scripts/scaling.jl` over the SuiteSparse/SNAP corpus (see the README).

Run with the project environment active: `julia --project=. -e 'using IJulia; notebook()'`, or execute
`julia --project=. examples/reproduce.jl` for the plain-script version.

In [ ]:
using DNLF, LinearAlgebra, SparseArrays, Printf, Random
reldiff(a,b) = norm(a-b)/max(norm(b),1e-30)
net = DNLF.load_tntp_net(joinpath(pkgdir(DNLF),"data","SiouxFalls","SiouxFalls_net.tntp"))
r,s,D = 1,20,3.0e4; d = zeros(net.n); d[r]=D; d[s]=-D
@printf("Sioux Falls: n=%d  m=%d\n", net.n, net.m)

## 1. Equilibrium correctness (near-linear approxChol vs. Frank–Wolfe)
Paper: relative flow difference ~1e-11.

In [ ]:
ffw = DNLF.frank_wolfe(net,r,s,D; iters=40000)
t = @elapsed ((φ,f,st) = DNLF.solve_ue(net,r,s,D; inner=:approxchol, tol=1e-10))
@printf("approxChol vs Frank–Wolfe: reldiff = %.2e   (%.3fs, %d Newton steps)\n", reldiff(f,ffw), t, st)

## 2. Adjoint design gradient vs. finite differences
Paper: max relative error ~2.6e-3 (finite-difference limited).

In [ ]:
φ,f,_ = DNLF.solve_ue(net,r,s,D; tol=1e-11, nmax=6000)
g = DNLF.adjoint_grad(net,s,φ,f,zeros(net.m)); ε=1e-2; active=findall(>(1e-6),f); mx=0.0
for a in active
    τp=zeros(net.m);τp[a]+=ε; _,fp,_=DNLF.solve_ue(net,r,s,D;tolls=τp,tol=1e-11,nmax=6000)
    τm=zeros(net.m);τm[a]-=ε; _,fm,_=DNLF.solve_ue(net,r,s,D;tolls=τm,tol=1e-11,nmax=6000)
    fd=(DNLF.tstt(net,fp)-DNLF.tstt(net,fm))/(2ε); global mx=max(mx,abs(g[a]-fd)/max(abs(fd),1.0))
end
@printf("adjoint max rel err vs finite differences (%d active arcs): %.2e\n", length(active), mx)

## 3. Bilevel toll design: monotone descent to the marginal-cost optimum
Paper: 5.15% TSTT reduction.

In [ ]:
φ,f,_,_ = DNLF.solve_flow(net,d,zeros(net.m); tol=1e-11); τ=zeros(net.m); T0=DNLF.tstt(net,f); T=T0
for _ in 1:40
    gv=DNLF.adjoint_grad(net,s,φ,f,τ); lr=0.05; acc=false
    for _ in 1:22
        τt=max.(τ.-lr.*gv,0.0); φt,ft,_,_=DNLF.solve_flow(net,d,τt;tol=1e-11,init=φ); Tt=DNLF.tstt(net,ft)
        if Tt<T; global τ=τt; global φ=φt; global f=ft; global T=Tt; acc=true; break; end; lr*=0.5
    end
    acc || break
end
@printf("TSTT %.1f -> %.1f  (%.2f%% reduction)\n", T0, T, 100*(1-T/T0))

The full irregular-graph crossover (near-linear vs. direct factorization) is `scripts/scaling.jl`:
```
AIER_DATA=~/code/aier/data julia --project=. scripts/scaling.jl SNAP__as-735 SNAP__Oregon-1 SNAP__as-caida
```